# Chapter 7 · 정책 Placebo

**연구질문:** 정책 효과 추정치는 시점·공간 위약검정의 분포에서 극단적인가?

> 위에서 아래로 실행하면 끝까지 동작합니다.

## 1. 핵심 개념·수식

**누수(leakage):** 예측 시점 $t$에 알 수 없는 정보가 특성·전처리·분할에 들어가 성능이 과대평가되는 문제. 대표 경로 세 가지.
1. **전처리 누수:** 표준화·결측대체·모수 추정을 전체 자료로 한다(학습구간 통계만 써야 함).
2. **미래 정보 누수:** 중심 이동평균 등 $y_{t+k}$를 포함하는 특성.
3. **분할 누수:** 시계열에 무작위 K-겹 교차검증을 쓴다(인접 관측이 학습·검증에 섞임).

**Walk-forward(롤링 오리진) 평가:** 시점 $T_1<T_2<\dots$마다 $[1,T_k]$로 학습하고 $(T_k,T_k+h]$를 평가한다. 폴드별 오차의 분포(평균·분산·최악)를 본다.

**Diebold–Mariano 검정:** 손실차 $d_t=L(e^{A}_t)-L(e^{B}_t)$의 평균이 0인지 검정한다. $\mathrm{DM}=\bar d/\sqrt{\widehat{\mathrm{LRV}}(d)/n}$.

**부트스트랩 신뢰구간:** 폴드 손실차를 블록 부트스트랩하여 우열의 불확실성을 본다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
# ---- 패널·인과추론용 최소제곱과 로버스트 표준오차 (수식이 코드에 그대로 보이도록 직접 구현) ----
def ols(y, X, names=None, cov="HC1", groups=None, maxlags=None):
    y = np.asarray(y, float).reshape(-1); X = np.asarray(X, float)
    n, k = X.shape
    XtX_inv = np.linalg.pinv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    if cov == "nonrobust":
        V = XtX_inv * (e @ e) / (n - k)
    elif cov == "HC1":
        V = XtX_inv @ (X.T * e ** 2 @ X) @ XtX_inv * n / (n - k)
    elif cov == "HAC":                                   # Newey-West, Bartlett 커널
        L = maxlags if maxlags is not None else int(np.floor(4 * (n / 100) ** (2 / 9)))
        Xe = X * e[:, None]
        S = Xe.T @ Xe
        for l in range(1, L + 1):
            G = Xe[l:].T @ Xe[:-l]
            S += (1 - l / (L + 1)) * (G + G.T)
        V = XtX_inv @ S @ XtX_inv * n / (n - k)
    elif cov == "cluster":                               # 군집 로버스트 (소표본 보정 포함)
        g = pd.factorize(np.asarray(groups))[0]; G = g.max() + 1
        S = np.zeros((k, k))
        for j in range(G):
            s = (X[g == j] * e[g == j][:, None]).sum(axis=0)
            S += np.outer(s, s)
        V = XtX_inv @ S @ XtX_inv * (G / (G - 1)) * ((n - 1) / (n - k))
    else:
        raise ValueError(cov)
    se = np.sqrt(np.clip(np.diag(V), 0, None))
    t = b / np.where(se > 0, se, np.nan)
    dof = (len(np.unique(groups)) - 1) if cov == "cluster" else (n - k)
    p = 2 * (1 - stats.t.cdf(np.abs(t), dof))
    names = names or [f"x{i}" for i in range(k)]
    return {"beta": b, "se": se, "t": t, "p": p, "resid": e, "V": V, "names": list(names), "n": n,
            "table": pd.DataFrame({"coef": b, "se": se, "t": t, "p": p}, index=names)}

def make_panel(N=24, T=48, start="2018-01-01", freq="MS", n_treated=3, T0=30, tau=-8.0, seed=SEED,
               effect_shape="step", parallel=False):
    # NOT real data: 요인모형 패널. 처치효과 tau를 알고 있으므로 추정기가 이를 회복하는지 확인할 수 있다.
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, periods=T, freq=freq)
    f1 = np.cumsum(rng.normal(0, 0.5, T)); f2 = np.sin(np.arange(T) * 2 * np.pi / 12)
    lam1 = rng.normal(1, 0.3, N); lam2 = rng.normal(0, 1, N); alpha = rng.normal(50, 8, N)
    if parallel:                                   # 모든 단위가 같은 공통추세만 가짐 -> 평행추세가 성립하는 설계
        lam1 = np.ones(N); lam2 = np.zeros(N)
    Y = alpha[:, None] + lam1[:, None] * f1[None] + 2 * lam2[:, None] * f2[None] + rng.normal(0, 1.0, (N, T))
    treated = list(range(n_treated))
    for i in treated:
        post = np.arange(T) >= T0
        ramp = np.clip((np.arange(T) - T0 + 1) / 6, 0, 1) if effect_shape == "ramp" else 1.0
        Y[i] += tau * post * ramp
    df = pd.DataFrame(Y.T, index=dates, columns=[f"U{i:02d}" for i in range(N)])
    return df, [f"U{i:02d}" for i in treated], T0

## 2. 데이터와 SCM 효과

In [ ]:
# Prop 99(실제) 또는 합성 패널로 SCM 효과와 시점·공간 위약분포를 비교한다.
def parse_prop99(raw):
    text = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else raw
    df = pd.read_csv(io.StringIO(text), sep=";"); cols = {c.strip().lower(): c for c in df.columns}
    for need in ("state", "year", "packspercapita", "treated"):
        if need not in cols: raise ValueError("예상 열 없음: " + need)
    return df.rename(columns={cols["state"]: "State", cols["year"]: "Year", cols["packspercapita"]: "PacksPerCapita", cols["treated"]: "treated"})

panel, treated, T0, DATA_KIND = None, None, None, None
if USE_LIVE_DATA:
    try:
        df99 = parse_prop99(fetch_bytes("https://raw.githubusercontent.com/synth-inference/synthdid/master/data/california_prop99.csv"))
        wide = df99.pivot(index="Year", columns="State", values="PacksPerCapita").sort_index()
        tr_state = df99.loc[df99["treated"] == 1, "State"].unique().tolist(); first_year = int(df99.loc[df99["treated"] == 1, "Year"].min())
        if wide.isna().any().any() or len(tr_state) != 1: raise ValueError("패널 불균형 또는 처치 단위 != 1")
        panel = wide[tr_state + [c for c in wide.columns if c not in tr_state]]; treated = tr_state
        T0 = int(np.where(panel.index == first_year)[0][0]); DATA_KIND = "REAL: California Prop 99 (synthdid 저장소)"
    except Exception as e:
        print("Prop 99 수집 실패 -> 합성 대체:", type(e).__name__, str(e)[:120])
if panel is None:
    panel, treated, T0 = make_panel(N=30, T=31, start="1970-01-01", freq="D", n_treated=1, T0=19, tau=-8.0, seed=SEED)
    panel.index = pd.Index(range(1970, 2001), name="Year"); DATA_KIND = "SYNTHETIC (NOT real data): 요인모형 패널, 주입 효과 tau=-8.0"
print("DATA_KIND:", DATA_KIND, "| 패널:", panel.shape, "| 처치:", treated, "| T0 위치:", T0)

def scm_weights(y1_pre, Y0_pre):
    J = Y0_pre.shape[1]; f = lambda w: np.sum((y1_pre - Y0_pre @ w) ** 2)
    return optimize.minimize(f, np.full(J, 1 / J), method="SLSQP", bounds=[(0, 1)] * J,
                             constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}], options={"maxiter": 300, "ftol": 1e-10}).x
def scm_effect(P, unit, donors, t0):
    y1 = P[unit].to_numpy(float); Y0 = P[donors].to_numpy(float); w = scm_weights(y1[:t0], Y0[:t0]); gap = y1 - Y0 @ w
    return gap[t0:].mean(), np.sqrt(np.mean(gap[:t0] ** 2)), np.sqrt(np.mean(gap[t0:] ** 2))
unit1 = treated[0]; donors = [c for c in panel.columns if c not in treated]
eff, rp, rq = scm_effect(panel, unit1, donors, T0)
print(f"SCM 사후 평균 효과 = {eff:.3f} | 사전 RMSPE = {rp:.3f}")

## 3. 시점·공간 위약

In [ ]:
# ---- 공간 위약: 각 도너를 가짜 처치로 ----
plac = []
for d_ in donors:
    e_, a_, b_ = scm_effect(panel, d_, [c for c in donors if c != d_], T0); plac.append((d_, e_, a_, b_))
P = pd.DataFrame(plac, columns=["unit", "effect", "rmspe_pre", "rmspe_post"]); P["ratio"] = P["rmspe_post"] / P["rmspe_pre"].clip(lower=1e-9)
ratio_t = rq / max(rp, 1e-9)
p_space = (1 + int((P["ratio"] >= ratio_t).sum())) / (1 + len(P))
print(f"[공간 위약] 처치 비율={ratio_t:.2f}, 위약 중앙값={P['ratio'].median():.2f}, 순열 p={p_space:.3f} (최소 가능 p={1/(1+len(P)):.3f})")
good = P[P["rmspe_pre"] <= 2 * max(rp, 1e-9)]
print(f"사전 적합이 나쁜 도너(사전 RMSPE>2배)를 제외하면 위약 {len(good)}개, p={(1 + int((good['ratio'] >= ratio_t).sum())) / (1 + len(good)):.3f}")
# ---- 시간 위약: 처치 이전 기간 안에서 가짜 개입 시점 ----
rows = []
for fake in range(max(8, T0 // 3), T0 - 2):
    e_, a_, b_ = scm_effect(panel.iloc[:T0], unit1, donors, fake); rows.append({"가짜 T0": panel.index[fake], "효과": e_})
Tp = pd.DataFrame(rows).set_index("가짜 T0"); show(Tp)
print(f"실제 효과 {eff:.2f} vs 시간 위약 효과 범위 [{Tp['효과'].min():.2f}, {Tp['효과'].max():.2f}]  -> 실제 효과가 위약 범위 밖이어야 설득력이 있다")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6)); ax.hist(P["effect"], bins=15, alpha=0.6, label="공간 위약 효과"); ax.axvline(eff, color="C3", lw=2, label="처치 단위"); ax.legend()
ax.set_title("효과 추정치의 위약 분포"); plt.show()

## 3. 결과 해석 작성
1. 정상 설정과 누수 설정의 성능 차이를 표로 제시하고, **어떤 누수 경로가 가장 큰 왜곡을 만들었는지** 설명한다.
2. 폴드(또는 지역·기간) 간 변동을 보고 "평균은 좋지만 최악의 경우가 나쁘다"를 구분한다.
3. 결론(모형 우열, 효과 방향)이 설정 변경에도 유지되는가? 유지되지 않으면 "강건하다"가 아니라 "민감하다"로 쓴다.
4. 한계: 폴드 수, 이벤트 수, 합성자료의 한계, 위약검정이 모든 식별 가정을 증명하지는 않는다는 점.

**제출:** 정상/누수 비교표, 폴드별 결과, 강건성 표, 한계 단락. 누수 점검 목록(`validation/Leakage_Checklist.md`)과 대조하라.